# Logic behind

According to the logic developed here, a venue is a hot venue if for each year more than n% (e.g.40%) of the athletes perform much better than their average in that year. 
 - Keep only the athlete that for each year they run, they have at least three races.
 - Average performance per athlete per year
 - For each race of an athlete in a specific venue  in a specific year (dataset row), compute the venue_gap between the average performance of that athlete in that year and the Mark he got in that venue.
 - Compute the lower and the upper fence of each venue_gap by Tukey's fences rule to identify the outlier. So I define that a venue is a hot or cold venue if it is an outlier in the venue_gap (so if in this venue there have been huge differences in performance) and if at least n% (e.g.40%) of the athletes experienced a "huge gap" in that venue. 


# Data collection

In [592]:
import numpy as np
import pandas as pd
import re

This dataset is the one corrected by the wind neutralization (Mureika formula). The column "CorrectedTime" contains the mark corrected from the wind

In [593]:

combined_df = pd.read_csv("./Data/100m_races_2015_2024_correctedwind.csv")

# Preview the data
combined_df.rename(columns={"CorrectedTime": "CorrectedTime_wind"}, inplace=True)
combined_df.head()

,WAId,Gender,FirstName,LastName,birthdate,Place,Race,ResultScore,Mark,Year,...,Category,ResultType,Country,Remark,Indoor,DisciplineCode,Discipline,Elevation,CorrectedTime_wind,CorrectedResultScore
0,14499703,Male,Michael,GOLDIE,07/03/1994,3.0,F2,922,10.88,2015,...,F,NaN,NZL,NaN,False,100,100 Metres,11.0,10.99,889.0
1,14499704,Male,Mathew,CONNOLLY,02/05/1995,4.0,F2,848,11.13,2015,...,F,NaN,NZL,NaN,False,100,100 Metres,11.0,11.25,814.0
2,14631002,Male,Ethan,HOLMAN,10/01/1999,6.0,F2,715,11.61,2015,...,F,NaN,NZL,NaN,False,100,100 Metres,11.0,11.74,681.0
3,14631003,Male,Daniel,HENDERSON,08/04/1996,5.0,F2,800,11.30,2015,...,F,NaN,NZL,NaN,False,100,100 Metres,11.0,11.42,766.0
4,14385185,Male,Cameron,FRENCH,17/05/1992,2.0,F2,1021,10.56,2015,...,F,NaN,NZL,NaN,False,100,100 Metres,11.0,10.67,986.0


In [594]:
combined_df.shape #we have 563698 races

(543541, 25)

# Filtering data

We have to keep only the athletes that for each year they run, they have at least three races, so that the average performance per year is reliable.

In [595]:
# 1. Calculate the number of races for each athlete per year
# transform('size') returns a Series with the same length as the original dataframe
group_counts = combined_df.groupby(['WAId', 'Year'])['WAId'].transform('size')

# 2. Keep only the rows where that specific year's count is 3 or more
combined_df = combined_df[group_counts >= 3].copy()
# 4. Optional: Verify the result
print(f"Original row count: {len(group_counts)}")
print(f"Filtered row count: {len(combined_df)}")

Original row count: 543541
Filtered row count: 419400


In [596]:
races_per_year = combined_df.groupby(['WAId', 'Year']).size().reset_index(name='Races_Count')

athlete_averages = races_per_year.groupby('WAId')['Races_Count'].mean().reset_index(name='Avg_Races_Per_Year')

print(athlete_averages.head(20))
print(f"On average athletes perform: {athlete_averages['Avg_Races_Per_Year'].mean()} per year")

        WAId  Avg_Races_Per_Year
0   14164651            4.333333
1   14165016            6.400000
2   14165060            4.000000
3   14165090            7.000000
4   14165216           11.000000
5   14165333            4.000000
6   14165400            3.000000
7   14165515            3.000000
8   14165717            4.000000
9   14165796            4.666667
10  14165879            3.000000
11  14165990            3.000000
12  14165991            8.500000
13  14166337            4.500000
14  14166368            9.000000
15  14166386            6.833333
16  14166526            3.000000
17  14166617            3.000000
18  14166621            3.000000
19  14166790            4.000000
On average athletes perform: 5.137757433971733 per year


In [597]:
def text_to_float_Mark(x):
    if pd.isna(x):
        return None
    
    x_str = str(x).strip()
    clean_value = re.findall(r"[-+]?\d*\.\d+|\d+", x_str)
    
    if clean_value:
        try:
            return float(clean_value[0])
        except ValueError:
            return None
    return None


combined_df['CorrectedTime_wind'] = combined_df['CorrectedTime_wind'].apply(lambda x: text_to_float_Mark(x))
combined_df['CorrectedTime_wind'].head()

0    10.99
2    11.74
3    11.42
5    10.66
6    11.50
Name: CorrectedTime_wind, dtype: float64

# Average Performance per Athlete per Year

In [598]:
combined_df['Athlete_Avg_Year'] = combined_df.groupby(['WAId', 'Year'])['CorrectedTime_wind'].transform('mean')
print(combined_df[['WAId', 'Year', 'CorrectedTime_wind', 'Athlete_Avg_Year']].head(20))

        WAId  Year  CorrectedTime_wind  Athlete_Avg_Year
0   14499703  2015               10.99         10.995556
2   14631002  2015               11.74         11.306364
3   14631003  2015               11.42         11.448000
5   14366795  2015               10.66         10.773750
6   14638321  2015               11.50         11.436000
7   14554527  2015               11.72         11.621111
8   14506688  2015               11.61         11.835000
9   14559004  2015               11.21         11.126667
10  14366795  2015               11.09         10.773750
11  14386819  2015               11.67         11.600000
12  14401722  2015               10.83         10.830000
13  14386821  2015               11.17         11.020000
14  14636402  2015               11.42         11.367857
16  14614622  2015               11.15         11.013333
18  14497633  2015               11.16         11.083750
19  14484220  2015               11.32         11.213333
21  14610243  2015             

In [599]:
combined_df['Athlete_Avg_Year'].shape

(419400,)

For example, for the athlete WAId=14631002 (Ethan HOLMAN)

In [600]:
combined_df[combined_df['WAId']==14631002]

,WAId,Gender,FirstName,LastName,birthdate,Place,Race,ResultScore,Mark,Year,...,ResultType,Country,Remark,Indoor,DisciplineCode,Discipline,Elevation,CorrectedTime_wind,CorrectedResultScore,Athlete_Avg_Year
2,14631002,Male,Ethan,HOLMAN,10/01/1999,6.0,F2,715,11.61,2015,...,NaN,NZL,NaN,False,100,100 Metres,11.0,11.74,681.0,11.306364
736,14631002,Male,Ethan,HOLMAN,10/01/1999,3.0,H3,814,11.25,2015,...,NaN,NZL,NaN,False,100,100 Metres,52.0,11.43,764.0,11.306364
2069,14631002,Male,Ethan,HOLMAN,10/01/1999,4.0,F,822,11.22,2015,...,NaN,NZL,NaN,False,100,100 Metres,11.0,11.30,800.0,11.306364
2096,14631002,Male,Ethan,HOLMAN,10/01/1999,2.0,H1,828,11.20,2015,...,NaN,NZL,NaN,False,100,100 Metres,11.0,11.48,750.0,11.306364
44300,14631002,Male,Ethan,HOLMAN,10/01/1999,1.0,NaN,848,11.13,2015,...,NaN,NZL,NaN,False,100,100 Metres,11.0,11.21,825.0,11.306364
44453,14631002,Male,Ethan,HOLMAN,10/01/1999,2.0,F4,755,11.46,2015,...,NaN,NZL,NaN,False,100,100 Metres,52.0,11.46,755.0,11.306364
44550,14631002,Male,Ethan,HOLMAN,10/01/1999,2.0,F,907,10.93,2015,...,NaN,NZL,NaN,False,100,100 Metres,11.0,11.14,845.0,11.306364
44634,14631002,Male,Ethan,HOLMAN,10/01/1999,1.0,QF4,913,10.91,2015,...,NaN,NZL,NaN,False,100,100 Metres,14.0,11.16,840.0,11.306364
44665,14631002,Male,Ethan,HOLMAN,10/01/1999,1.0,H7,854,11.11,2015,...,NaN,NZL,NaN,False,100,100 Metres,14.0,11.11,854.0,11.306364
44794,14631002,Male,Ethan,HOLMAN,10/01/1999,1.0,SF2,825,11.21,2015,...,NaN,NZL,NaN,False,100,100 Metres,14.0,11.19,831.0,11.306364


# Venue Gap

In [601]:
combined_df['Venue_Gap'] = combined_df['Athlete_Avg_Year'] - combined_df['CorrectedTime_wind']

combined_df[['WAId', 'CorrectedTime_wind', 'Athlete_Avg_Year', 'Venue', 'Venue_Gap']].head(20)

,WAId,CorrectedTime_wind,Athlete_Avg_Year,Venue,Venue_Gap
0,14499703,10.99,10.995556,Tauranga (NZL),0.005556
2,14631002,11.74,11.306364,Tauranga (NZL),-0.433636
3,14631003,11.42,11.448000,Tauranga (NZL),0.028000
5,14366795,10.66,10.773750,Tauranga (NZL),0.113750
6,14638321,11.50,11.436000,Timaru (NZL),-0.064000
7,14554527,11.72,11.621111,Hobart (AUS),-0.098889
8,14506688,11.61,11.835000,Hobart (AUS),0.225000
9,14559004,11.21,11.126667,Timaru (NZL),-0.083333
10,14366795,11.09,10.773750,Tauranga (NZL),-0.316250
11,14386819,11.67,11.600000,Canberra (AUS),-0.070000


# Let's find Hot or Cold venues

So I define that a venue is a hot or cold venue if it is an outlier in the mark_gap (so if in this venue there have been huge differences in performance) and if there are at least 40% of the athletes experienced a "huge gap" (in the outlier) in that venue. 

Outliers are found by Tukey's Fences rule (we make the rule more conservative considerig the Gap of (1.1 * IQR) instead of (1.5 * IQR):

$$Lower\ Fence = Q_1 - 1.1 \times IQR$$

$$Upper\ Fence = Q_3 + 1.1 \times IQR$$

Any value $< \text{Lower Fence}$ or $> \text{Upper Fence}$ is an outlier.

In [602]:
# 1. Calculate IQR for the Venue_Gap to find Tukey's Fences
Q1 = combined_df['Venue_Gap'].quantile(0.25)
Q3 = combined_df['Venue_Gap'].quantile(0.75)
IQR = Q3 - Q1

# Define the Fences
upper_fence = Q3 + (1.1 * IQR)  # Threshold for HOT venues (Positive Gap)
lower_fence = Q1 - (1.1 * IQR)  # Threshold for COLD venues (Negative Gap)

print(f"Hot Outlier Threshold (Gap > {upper_fence:.3f})")
print(f"Cold Outlier Threshold (Gap < {lower_fence:.3f})")

# 1. Identify which individual races are "Outlier Performances" 
# (Assuming Q1, Q3, and IQR were calculated from Venue_Gap as in your snippet)
combined_df['Is_Hot_Outlier'] = combined_df['Venue_Gap'] > upper_fence
combined_df['Is_Cold_Outlier'] = combined_df['Venue_Gap'] < lower_fence

# 2. Aggregate by Venue to calculate percentages
# We need both the number of outliers AND the total unique athletes at the venue
venue_stats = combined_df.groupby(['Venue', 'Country']).agg(
    total_unique_athletes=('WAId', 'nunique'),
    unique_hot_athletes=('WAId', lambda x: combined_df.loc[x.index, 'WAId'][combined_df.Is_Hot_Outlier].nunique()),
    unique_cold_athletes=('WAId', lambda x: combined_df.loc[x.index, 'WAId'][combined_df.Is_Cold_Outlier].nunique()),
    avg_venue_gap=('Venue_Gap', 'mean')
).reset_index()

# 3. Calculate the percentage of athletes who experienced a "huge gap"
venue_stats['pct_hot_athletes'] = venue_stats['unique_hot_athletes'] / venue_stats['total_unique_athletes']
venue_stats['pct_cold_athletes'] = venue_stats['unique_cold_athletes'] / venue_stats['total_unique_athletes']

# 4. Apply your new logic: At least 40% of athletes must be outliers and 'total_unique_athletes >= 5' to avoid 1-person venues)
hot_venues = venue_stats[(venue_stats['pct_hot_athletes'] >= 0.4) & 
    (venue_stats['total_unique_athletes'] >= 5)].sort_values(by='pct_hot_athletes', ascending=False)
cold_venues = venue_stats[(venue_stats['pct_cold_athletes'] >= 0.4) & 
    (venue_stats['total_unique_athletes'] >= 5)].sort_values(by='pct_cold_athletes', ascending=False)

Hot Outlier Threshold (Gap > 0.214)
Cold Outlier Threshold (Gap < -0.205)


In [603]:
# 1. Get unique lists of names
hot_names = set(hot_venues['Venue'].unique())
cold_names = set(cold_venues['Venue'].unique())

# 2. If a venue is in both, it's 'Neutral' because the data is contradictory.
conflicting = hot_names.intersection(cold_names)
clean_hot = hot_names - conflicting
clean_cold = cold_names - conflicting

# 3. Apply labels strictly
combined_df['Is_Hot_Venue'] = combined_df['Venue'].isin(clean_hot)
combined_df['Is_Cold_Venue'] = combined_df['Venue'].isin(clean_cold)

In [604]:
print("-----Hot venues-----")
print(combined_df["Venue"][combined_df['Venue'].isin(clean_hot)].unique())
print(f"Hot Venues identified: {len(clean_hot)}")

-----Hot venues-----
['Yukon, OK (USA)' 'Panama City (PAN)' 'Lodi, CA (USA)' 'Brest (FRA)'
 'Cannington (AUS)']
Hot Venues identified: 5


In [605]:
print("-----Cold venues-----")
print(combined_df["Venue"][combined_df['Venue'].isin(clean_cold)].unique())
print(f"Cold Venues identified: {len(clean_cold)}")

-----Cold venues-----
['Magglingen (SUI) (i)' 'Itajaí (BRA)' 'Grimstad (NOR)' 'Bonneville (FRA)'
 'Närpiö (FIN)' 'Sertaozinho (BRA)' 'Athens, OH (USA)' 'Leicester (GBR)'
 'St-Pol-sur-Ternoise (FRA)' 'Plattling (GER)' 'Valparaíso (CHI)'
 'Visby (SWE)' 'Ostroda (POL)' 'Bullard, TX (USA)' 'Geithus (NOR)'
 'Leduc (CAN)' 'Paralimni (CYP)' 'Brodnica (POL)' 'Lexington, VA (USA)'
 'Mountain Brooks, AL (USA)' 'Telford (GBR)' 'Raisio (FIN)' 'Čačak (SRB)'
 'Fuji (JPN)' 'Panevėžys (LTU)' 'Orléans (FRA)' 'Cheltenham (GBR)'
 'North Newton, KS (USA)' 'Ningbo (CHN)' 'Misiones (ARG)' 'Namboole (UGA)'
 'Aracaju (BRA)' 'Zwijndrecht (BEL)' 'Guarapuava (BRA)']
Cold Venues identified: 34


# Let's correct the impact of these venues for neutralization

To neutralize the effect of hot/cold venues, we could adjust the athlete's mark ($M$) in these venues by the venue's specific average gap, doing this the correction is not done by a fix value but it is done based on the venue; based on how much different athletes on average perform much better or much worse than their average in that venue

- Hot Venues:
$$M_{corrected} = M_{original} + |\text{avg\_venue\_gap}|$$

- Cold Venues:
$$M_{corrected} = M_{original} - |\text{avg\_venue\_gap}|$$

In [606]:
# 4. Re-build the correction dictionary only for non Neutral venues
final_significant = venue_stats[
    venue_stats['Venue'].isin(clean_hot | clean_cold)
].drop_duplicates(subset=['Venue'])

correction_lookup = final_significant.set_index('Venue')['avg_venue_gap'].to_dict()

# 5. Apply the Map
combined_df['venue_correction_gap'] = combined_df['Venue'].map(correction_lookup)

# 6. Apply Split Correction Logic
combined_df['CorrectedTime_venue'] = np.select(
    [
        combined_df['Is_Hot_Venue'], 
        combined_df['Is_Cold_Venue']
    ],
    [
        combined_df['CorrectedTime_wind'] + combined_df['venue_correction_gap'].abs(), # Hot: Subtract abs
        combined_df['CorrectedTime_wind'] - combined_df['venue_correction_gap'].abs()  # Cold: Add abs
    ],
    default=combined_df['CorrectedTime_wind'] # Neutral: No change
)

# 7. Verification
print(f"Rows where Mark was changed: {(combined_df['CorrectedTime_wind'] != combined_df['CorrectedTime_venue']).sum()}")

Rows where Mark was changed: 643


In [607]:
#corrected_dataset = combined_df.copy()

# Export the final corrected dataset to a CSV file
#corrected_dataset.to_csv('athlete_venue_corrections.csv', index=False)